In [ ]:
# @title 1. Setup Determinístico (Qwen3-TTS + WhisperX + Fix NumPy)
import os
from google.colab import drive

print("🔌 Montando Google Drive corporativo...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("📦 Instalando dependências nativas de sistema (C/C++)...")
!apt-get update -qq && apt-get install -qq -y portaudio19-dev espeak-ng ffmpeg

print("🐍 Instalando SDK oficial do Qwen3-TTS e manipulação de áudio...")
!pip install -q -U qwen-tts soundfile scipy

print("🗣️ Instalando WhisperX para alinhamento de precisão por palavra...")
!pip install -q git+https://github.com/m-bain/whisperx.git

print("🛠️ Resolvendo Conflito Arquitetural: Ajustando NumPy para Versão de Equilíbrio (2.0.2)...")
# NumPy 2.0.2 é o ponto de compatibilidade entre Numba (<=2.0) e as libs modernas do Colab.
!pip install -qU "numpy==2.0.2"

print("\n✅ Ambiente configurado! REINICIE A SESSÃO (Ambiente de Execução -> Reiniciar sessão) e execute a Célula 2.")

🔌 Montando Google Drive corporativo...
Mounted at /content/drive
📦 Instalando dependências nativas de sistema (C/C++)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
dpkg: libjack-jackd2-0:amd64: dependency problems, but removing anyway as you requested:
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.
 libavdevice58:amd64 depends on libjack-jackd2-0 (>= 1.9.10+20150825) | libjack-0.125; however:
  Package libjack-jackd2-0:amd64 is to be removed.
  Package libjack-0.125 is not installed.
  Package libjack-jackd2-0:amd64 which provides libjack-0.125 is to be removed.

(Reading database ... 122403 files and directories currently i

In [ ]:
# @title 2. Pipeline de Geração (Qwen3-TTS VoiceDesign + WhisperX Sync)
import torch
import os
import json
import whisperx
import re
import soundfile as sf
import numpy as np
import shutil
import random
import glob
from datetime import datetime
from qwen_tts import Qwen3TTSModel

# --- 1. Topologia de Diretórios ---
BASE_DIR = "/content/drive/MyDrive/nexus_pipeline"
STAGING_DIR = os.path.join(BASE_DIR, "staging")
AUDIO_READY_DIR = os.path.join(BASE_DIR, "audio_ready")
LOCAL_SCRATCH = "/content/scratch_pipeline"

# --- 2. Engenharia de Voz ---
VOICE_DESCRIPTION = (
    "A calm, stable, and neutral male voice acting as a professional documentary presenter. "
    "The tone is objective and factual, with a steady and controlled pace."
)
LANGUAGE = "English"
FIXED_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

def split_text_into_chunks(text, max_words=80):
    sentences = re.split(r'(?<=[.!?])\\s+', text)
    chunks = []
    current_chunk = []
    current_words = 0
    for sentence in sentences:
        words_in_sentence = len(sentence.split())
        if current_words + words_in_sentence > max_words:
            if current_chunk: chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_words = words_in_sentence
        else:
            current_chunk.append(sentence)
            current_words += words_in_sentence
    if current_chunk: chunks.append(" ".join(current_chunk))
    return chunks

def normalize_audio(audio_array):
    max_val = np.abs(audio_array).max()
    if max_val > 0: return audio_array / max_val * 0.9
    return audio_array

# --- 3. Fluxo Principal ---
os.makedirs(STAGING_DIR, exist_ok=True)
os.makedirs(AUDIO_READY_DIR, exist_ok=True)
os.makedirs(LOCAL_SCRATCH, exist_ok=True)

if not os.listdir(STAGING_DIR):
    print(f"⚠️ Staging vazio: {STAGING_DIR}")
else:
    print("⚖️ Alocando pesos Qwen3...")
    tts_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float16)
    print("[WhisperX] Inicializando motor...")
    whisper_model = whisperx.load_model("base", "cuda", compute_type="float16")
    align_models_cache = {}

    # Busca recursiva para encontrar scripts em subpastas
    all_txt_files = glob.glob(os.path.join(STAGING_DIR, "**/*.txt"), recursive=True)

    for txt_path in all_txt_files:
        # Determina a estrutura de pastas relativa para replicar no output
        rel_path = os.path.relpath(txt_path, STAGING_DIR)
        path_parts = rel_path.split(os.sep)

        # factory = primeira pasta, scene_id = nome do arquivo sem extensão
        factory = path_parts[0]
        scene_id = os.path.splitext(os.path.basename(txt_path))[0]

        # Se o arquivo estiver dentro de subpastas, incluímos elas no caminho de destino
        sub_dirs = os.sep.join(path_parts[1:-1])
        dest_dir = os.path.join(AUDIO_READY_DIR, factory, sub_dirs, scene_id)
        local_scene_dir = os.path.join(LOCAL_SCRATCH, scene_id)

        os.makedirs(dest_dir, exist_ok=True)
        os.makedirs(local_scene_dir, exist_ok=True)

        local_wav_path = os.path.join(local_scene_dir, f"{scene_id}.wav")
        local_json_path = os.path.join(local_scene_dir, "words.json")
        final_wav_path = os.path.join(dest_dir, f"{scene_id}.wav")
        final_json_path = os.path.join(dest_dir, f"{datetime.now().strftime('%Y%m%d%H%M%S')}_words.json")

        print(f"\n🎬 Processando: {rel_path}")
        with open(txt_path, 'r', encoding='utf-8') as f: text = f.read().strip()

        set_seed(FIXED_SEED)
        chunks = split_text_into_chunks(text, max_words=80)
        waveforms, success, sample_rate = [], True, None

        for i, chunk_text in enumerate(chunks):
            try:
                wavs, sr = tts_model.generate_voice_design(text=chunk_text, language=LANGUAGE, instruct=VOICE_DESCRIPTION)
                waveforms.append(wavs[0])
                sample_rate = sr
            except Exception as e:
                print(f"      ❌ Erro no lote {i+1}: {e}")
                success = False; break

        if success and waveforms:
            combined_wav = normalize_audio(np.concatenate(waveforms))
            sf.write(local_wav_path, combined_wav, sample_rate)
            try:
                audio = whisperx.load_audio(local_wav_path)
                result = whisper_model.transcribe(audio, batch_size=16)
                lang = result["language"]
                model_a, metadata = align_models_cache.get(lang, whisperx.load_align_model(language_code=lang, device="cuda"))
                align_models_cache[lang] = (model_a, metadata)
                result = whisperx.align(result["segments"], model_a, metadata, audio, "cuda")
                words = [{"word": w["word"], "start": w["start"], "end": w["end"]} for s in result["segments"] for w in s.get("words", []) if "start" in w]
                with open(local_json_path, 'w', encoding='utf-8') as f: json.dump(words, f, indent=2, ensure_ascii=False)
                shutil.move(local_wav_path, final_wav_path)
                shutil.move(local_json_path, final_json_path)
                print(f"✅ SUCESSO: {final_wav_path}")
            except Exception as e: print(f"   ❌ Erro WhisperX: {e}")

        if os.path.exists(local_scene_dir): shutil.rmtree(local_scene_dir)

    print("\n🏁 Concluído.")


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
⚖️ Alocando pesos Qwen3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

...

2026-06-14 15:39:22 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:01<00:00, 272MB/s]


✅ SUCESSO: /content/drive/MyDrive/nexus_pipeline/audio_ready/ensaio/preview_1min/preview_1min.wav

🎬 Processando: ensaio/full_documentary/script.txt


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


2026-06-14 16:09:31 - whisperx.asr - INFO - Detected language: en (1.00) in first 30s of audio
✅ SUCESSO: /content/drive/MyDrive/nexus_pipeline/audio_ready/ensaio/full_documentary/script/script.wav

🏁 Concluído.


In [ ]:
import IPython.display as ipd
import glob

# Busca todos os arquivos .wav na pasta de saída
audio_files = glob.glob(f"{AUDIO_READY_DIR}/**/*.wav", recursive=True)

if not audio_files:
    print("ℹ️ Nenhum áudio encontrado em:", AUDIO_READY_DIR)
else:
    print(f"🎧 Encontrados {len(audio_files)} arquivos de áudio:")
    for path in sorted(audio_files):
        rel_path = os.path.relpath(path, AUDIO_READY_DIR)
        print(f"\n▶️ Tocando: {rel_path}")
        ipd.display(ipd.Audio(path))

In [ ]:
import os

# Diagnostic: Check the specific path mentioned by the user
target_file = '/content/drive/MyDrive/nexus_pipeline/staging/ensaio/full_documentary/script.txt'

print(f"🔍 Checking for: {target_file}")
if os.path.exists(target_file):
    print("✅ File exists.")
    print(f"📏 Size: {os.path.getsize(target_file)} bytes")
else:
    print("❌ File NOT found at this path.")

print("\n📂 Current structure of STAGING_DIR:")
for root, dirs, files in os.walk(STAGING_DIR):
    level = root.replace(STAGING_DIR, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

🔍 Checking for: /content/drive/MyDrive/nexus_pipeline/staging/ensaio/full_documentary/script.txt
✅ File exists.
📏 Size: 11656 bytes

📂 Current structure of STAGING_DIR:
staging/
    the_sleep_protocol/
    ensaio/
        full_documentary/
            script.txt
